# Volume 14, Chapter 10: Primes to large moduli (Python companion)

Exercises first, solutions at the end. We measure how far one must walk along a progression to meet a prime, look at the smoothed sieve weights $\beta_R$ that make the log-free density estimate work, compute $L(1,\chi)$ for thousands of real characters, and watch the smoothed prime number theorem for progressions hold, then break, as $q$ grows towards $x$. The computer science application: **primes for the number theoretic transform**, primes $p=c\cdot2^k+1$ that make fast polynomial multiplication exact. Requires only `numpy`.

In [ ]:
import math, time
import numpy as np

def small_primes(n):
    s = np.ones(n + 1, dtype=bool); s[:2] = False
    for i in range(2, int(n ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]

## Exercises

**P10.1 (★) Least primes in progressions.** For every prime $q\le3000$ and every $a\not\equiv0$, find the least prime $p(q,a)\equiv a\pmod q$, and let $P(q)=\max_ap(q,a)$. Find the largest value of $\frac{\log P(q)}{\log q}$ and of $\frac{P(q)}{q^2}$, and compare $P(q)$ with $q\log^2q$, the size suggested by a random model (Figure 10.1).

**P10.2 (★★) Sieve weights.** With $f$ smooth, equal to $1$ on $(-\infty,1]$ and to $0$ on $[2,\infty)$, compute $\beta_R(n)=\sum_{d\mid n}\mu(d)f(\frac{\log d}{\log R})$ for $n\le10^6$ and $R=10,30,100$. Check that $\beta_R(n)=0$ for $1<n\le R$ and $\beta_R(p)=1$ for primes $p>R^2$, and compare $\sum_{n\le x}\beta_R(n)^2$ with $\frac x{\log R}$ (Exercise 10.1).

**P10.3 (★★) Siegel's $L(1,\chi)$, numerically.** For every prime $p$ with $5\le p<2\cdot10^5$, compute $L(1,\chi_p)$, where $\chi_p$ is the Legendre symbol, with Dirichlet's exact formulas: $L(1,\chi_p)=-\frac\pi{p^{3/2}}\sum_{a<p}a\chi_p(a)$ for $p\equiv3\pmod4$, and $L(1,\chi_p)=-\frac1{\sqrt p}\sum_{a<p}\chi_p(a)\log\sin\frac{\pi a}p$ for $p\equiv1\pmod4$. Check the class numbers $h(-p)=\frac{\sqrt p}\pi L(1,\chi_p)$ for $p=7,23,47,163$. How small does $L(1,\chi_p)$ get? Compare with $\frac1{\log\log p}$, the lower bound under GRH (Littlewood), and with Siegel's $p^{-\varepsilon}$.

**P10.4 (★★) Gallagher's range, seen.** With $w(t)=\exp(-\frac1{(t-\frac12)(2-t)})$ on $(\frac12,2)$, compute $\psi_w(x;q,a)=\sum_{n\equiv a\,(q)}\Lambda(n)w(\frac nx)$ for $x=10^7$ and prime moduli $q\approx10,10^2,\dots,10^6$, and the largest and median relative deviation from $\frac{x\hat w(1)}{q-1}$ over the classes. Relate the results to $\frac{\log x}{\log q}$ (Theorem 10.10).

**P10.5 (★, CS application) NTT primes.** The number theoretic transform multiplies polynomials with integer coefficients modulo a prime $p$ with $2^k\mid p-1$, so that $\mathbb Z/p\mathbb Z$ contains a primitive $2^k$-th root of unity. For $k\le62$, find the least $c$ with $c\cdot2^k+1$ prime (Linnik's question for the modulus $2^k$ and the class $1$), and compare $c$ with $\frac k2\log2$, the waiting time predicted by the prime number theorem for progressions. Then implement the NTT modulo $998244353=119\cdot2^{23}+1$ (primitive root $3$) and check a product of two random polynomials of degree $1000$ against direct multiplication.

## Solutions

### P10.1

In [ ]:
P = small_primes(10 ** 7)
res = []
for q in P[(P >= 3) & (P <= 3000)]:
    q = int(q)
    u, idx = np.unique(P % q, return_index=True)   # first prime in each class
    first = P[idx][u != 0]
    assert len(first) == q - 1
    res.append((q, int(first.max()), int(u[u != 0][np.argmax(first)])))
ratio = [math.log(Pq) / math.log(q) for q, Pq, a in res]
i = int(np.argmax(ratio))
print("largest log P/log q:", round(ratio[i], 3), "at q, P(q), worst class =", res[i])
print("largest P/q^2:", round(max(Pq / q ** 2 for q, Pq, a in res), 3))
model = np.array([Pq / (q * math.log(q) ** 2) for q, Pq, a in res])
print("P(q)/(q log^2 q): mean", round(model.mean(), 3), " min", round(model.min(), 3), " max", round(model.max(), 3))
for q, Pq, a in res:
    if q in (101, 1009, 2999):
        print(f"q={q}: P(q)={Pq} in class {a},  log P/log q = {math.log(Pq) / math.log(q):.3f}")

The worst case is $q=5$: the class $4$ waits for $19$, and $\frac{\log19}{\log5}=1.83$. Every other prime $q\le3000$ does better, and $P(q)<q^2$ throughout, as the conjecture $L=2$ predicts. The random model fits remarkably: the mean of $\frac{P(q)}{q\log^2q}$ is $1.03$. Heuristically, a class receives a prime about every $\varphi(q)\log x$ integers, and the last of $\varphi(q)$ independent geometric waiting times exceeds the mean by a factor $\log\varphi(q)$, hence $q\log^2q$. Linnik's theorem proves $P(q)\le q^5$; GRH would give $q^{2+\varepsilon}$; the truth seems to be $q^{1+o(1)}$.

### P10.2

In [ ]:
def f(u):
    # smooth, 1 on (-inf,1], 0 on [2,inf)
    u = np.asarray(u, float); out = np.zeros_like(u); out[u <= 1] = 1
    m = (u > 1) & (u < 2); v = u[m] - 1
    a, b = np.exp(-1 / v), np.exp(-1 / (1 - v)); out[m] = b / (a + b)
    return out

X = 10 ** 6
mu = np.ones(X + 1, dtype=int); isp = np.ones(X + 1, dtype=bool); isp[:2] = False
for p in range(2, X + 1):
    if isp[p]:
        isp[p * p::p] = False; mu[::p] *= -1; mu[::p * p] = 0
for R in [10, 30, 100]:
    beta = np.zeros(X + 1)
    ds = np.arange(1, R * R + 1); ds = ds[mu[ds] != 0]
    for d, fd in zip(ds, f(np.log(ds) / math.log(R))):
        beta[d::d] += mu[d] * fd
    big_primes = np.nonzero(isp[R * R + 1:])[0] + R * R + 1
    S = beta[1:] @ beta[1:]
    print(f"R={R}: max|beta| on (1,R] = {np.abs(beta[2:R + 1]).max()},  beta=1 on primes > R^2: {np.allclose(beta[big_primes], 1)},"
          f"  sum beta^2 / (x/log R) = {S / (X / math.log(R)):.3f},  max|beta| = {np.abs(beta[1:]).max():.2f}")

The two properties that make $\beta_R$ a sieve hold exactly: $\beta_R$ vanishes on $(1,R]$ and is $1$ on the primes above $R^2$. And $\sum_{n\le x}\beta_R(n)^2$ is a bounded multiple of $\frac x{\log R}$ (ratios $0.74$, $0.92$, $1.06$), although the individual weights reach $4$ or $5$: the weight is concentrated on the integers without small prime factors, a set of density about $\frac1{\log R}$. (For $R=100$ the range $x\ge R^4$ of Exercise 10.1 is not reached, but the bound still holds.) This is the saving of a logarithm that the log-free estimate needs.

### P10.3

In [ ]:
ps = small_primes(200000); ps = ps[ps >= 5]
Ls = np.empty(len(ps))
for i, p in enumerate(ps):
    p = int(p); a = np.arange(1, p)
    sq = np.zeros(p, dtype=bool); sq[(a * a) % p] = True
    chi = np.where(sq[a], 1.0, -1.0)
    if p % 4 == 3:
        Ls[i] = -math.pi / p ** 1.5 * np.dot(a, chi)
    else:
        Ls[i] = -np.dot(chi, np.log(np.sin(math.pi * a / p))) / math.sqrt(p)
for p in (7, 23, 47, 163):
    print(f"h(-{p}) = {math.sqrt(p) * Ls[ps == p][0] / math.pi:.6f}")
for lo, hi in [(5, 1e3), (1e3, 1e4), (1e4, 1e5), (1e5, 2e5)]:
    m = (ps >= lo) & (ps < hi); j = np.argmin(Ls[m]); p = int(ps[m][j])
    print(f"p in [{lo:.0e},{hi:.0e}): min L(1) = {Ls[m][j]:.4f} at p={p} (p mod 4 = {p % 4}),"
          f"  min*loglog p = {Ls[m][j] * math.log(math.log(p)):.3f},  mean = {Ls[m].mean():.3f},  max = {Ls[m].max():.3f}")

The class numbers come out as integers: $h(-7)=1$, $h(-23)=3$, $h(-47)=5$, $h(-163)=1$. The smallest value of $L(1,\chi_p)$ below $2\cdot10^5$ is $0.2235$, at $p=166147$; in every range, the minimum times $\log\log p$ stays between $0.40$ and $0.58$, as GRH predicts (it implies $L(1,\chi)\gg\frac1{\log\log q}$). Nothing like $p^{-\varepsilon}$ is in sight: an exceptional zero, if one exists, has not shown itself, and the mean of $L(1,\chi_p)$ settles near $1.6$. Siegel's theorem is a theorem about characters we will never compute, which is also why its constant cannot be computed.

### P10.4

In [ ]:
x = 10 ** 7
ps = small_primes(2 * x)
n = [ps]; lam = [np.log(ps).astype(float)]
for k in range(2, 25):                         # prime powers p^k <= 2x
    pk = np.array([int(p) ** k for p in ps[:1000] if int(p) ** k <= 2 * x], dtype=np.int64)
    if len(pk) == 0: break
    n.append(pk); lam.append(np.log(pk) / k)
n = np.concatenate(n); lam = np.concatenate(lam)

def w(t):
    t = np.asarray(t, float); out = np.zeros_like(t); m = (t > 0.5) & (t < 2)
    out[m] = np.exp(-1 / ((t[m] - 0.5) * (2 - t[m]))); return out
tt = np.linspace(0.5, 2, 300001); w1 = np.trapezoid(w(tt), tt)   # \hat w(1)
W = lam * w(n / x)
print(f"psi_w(x)/x = {W.sum() / x:.6f},  hat w(1) = {w1:.6f}")
for q in [11, 101, 1009, 10007, 100003, 1000003]:
    S = np.bincount(n % q, weights=W, minlength=q)[1:]
    dev = np.abs(S / (x * w1 / (q - 1)) - 1)
    print(f"q={q:8d}  log x/log q = {math.log(x) / math.log(q):5.2f}  max dev = {dev.max():7.4f}  median dev = {np.median(dev):.4f}  empty classes = {np.sum(S == 0)}")

The prime number theorem with the smooth weight is very accurate: $\psi_w(x)/x=0.129475$ against $\hat w(1)=0.129476$. For $q$ up to $10^3$ ($\frac{\log x}{\log q}\ge2.3$) every class is within about $7\%$ of its share. At $q=10^4$, where $\frac{\log x}{\log q}=1.75$, the worst class is off by $35\%$; at $q=10^5$ the median class is off by $22\%$ and one class receives no prime at all in $(\frac x2,2x)$; at $q=10^6$ most classes are empty, since there are only about $10^6$ primes in the window for $10^6$ classes. Theorem 10.10 promises good behaviour only when $\frac{\log x}{\log q}$ is a large constant, and the experiment shows that the transition happens around $\frac{\log x}{\log q}\approx2$, as GRH (and the square-root size of the error) suggests.

### P10.5

In [ ]:
def is_prime(n):
    # deterministic Miller-Rabin for n < 3.3e24
    if n < 2: return False
    small = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41)
    for p in small:
        if n % p == 0: return n == p
    d, s = n - 1, 0
    while d % 2 == 0: d //= 2; s += 1
    for a in small:
        y = pow(a, d, n)
        if y in (1, n - 1): continue
        for _ in range(s - 1):
            y = y * y % n
            if y == n - 1: break
        else: return False
    return True

rows = []
for k in range(1, 63):
    c = 1
    while not is_prime(c * 2 ** k + 1): c += 1
    rows.append((k, c))
print(" ".join(f"{k}:{c}" for k, c in rows))
print("mean of c/((k/2) log 2) for k>=6:", round(np.mean([c / (k / 2 * math.log(2)) for k, c in rows[5:]]), 3))
print("largest log p/log 2^k for k>=3:", round(max(math.log(c * 2 ** k + 1) / (k * math.log(2)) for k, c in rows[2:]), 3))
print("least c for k=23:", dict(rows)[23], "->", dict(rows)[23] * 2 ** 23 + 1)

In [ ]:
MOD, G = 998244353, 3
def ntt(a, invert=False):
    a = list(a); n = len(a); j = 0
    for i in range(1, n):                      # bit-reversal permutation
        bit = n >> 1
        while j & bit: j ^= bit; bit >>= 1
        j |= bit
        if i < j: a[i], a[j] = a[j], a[i]
    length = 2
    while length <= n:
        wl = pow(G, (MOD - 1) // length, MOD)
        if invert: wl = pow(wl, MOD - 2, MOD)
        for i in range(0, n, length):
            wn = 1
            for k in range(i, i + length // 2):
                u, v = a[k], a[k + length // 2] * wn % MOD
                a[k], a[k + length // 2] = (u + v) % MOD, (u - v) % MOD
                wn = wn * wl % MOD
        length <<= 1
    if invert:
        ninv = pow(n, MOD - 2, MOD); a = [v * ninv % MOD for v in a]
    return a

rng = np.random.default_rng(10)
A = rng.integers(0, 1000, 1001).tolist(); B = rng.integers(0, 1000, 1001).tolist()
size = 1
while size < len(A) + len(B) - 1: size *= 2
fa, fb = ntt(A + [0] * (size - len(A))), ntt(B + [0] * (size - len(B)))
prod = ntt([u * v % MOD for u, v in zip(fa, fb)], invert=True)[:len(A) + len(B) - 1]
direct = np.convolve(np.array(A, dtype=object), np.array(B, dtype=object)).tolist()
print("transform length", size, " exact:", prod == direct, " largest coefficient", max(direct))

The least $c$ jumps around ($c=1$ for $k=1,2,4,8,16$, the Fermat primes; $c=54$ for $k=58$), and on average it is about $1.18\cdot\frac k2\log2$, close to the prediction: among the numbers $c\cdot2^k+1$, which are all odd, a proportion $\frac2{\log(c2^k)}$ is prime. So primes $\equiv1\pmod{2^k}$ appear well before $(2^k)^2$, far below Linnik's bound $(2^k)^5$; the largest exponent $\frac{\log p}{\log2^k}$ for $k\ge3$ is $1.43$. The popular prime $998244353=119\cdot2^{23}+1$ is not the least for $k=23$ (that is $20\cdot2^{23}+1=167772161$); it is chosen because it is just below $2^{30}$, so that sums of two residues fit in a signed 32-bit integer. The product of two degree $1000$ polynomials is exact, because every coefficient of the true product is below $1000\cdot1000\cdot1001<998244353$.